# **Metrics**

Tests for the evaluation metrics in `metrics.py`:
- `get_ingredient_utilisation_score`
- `get_expiry_weighted_utilisation_score`
- `get_expiry_waste_score`
- `get_dietary_compliance`
- `get_budget_efficiency`
- `get_purchase_dependency_score`
- `get_variety_score`

In [ ]:
import sys
from datetime import datetime

sys.path.append("../..")

from metrics import (
    get_budget_efficiency,
    get_dietary_compliance,
    get_expiry_waste_score,
    get_expiry_weighted_utilisation_score,
    get_ingredient_utilisation_score,
    get_purchase_dependency_score,
    get_variety_score,
)

from engines import make_pantry, make_preferences, make_recipe
from models import DietaryTag

## 1. `get_ingredient_utilisation_score`

Measures what fraction of available pantry stock is consumed by the meal plan (0 - 1).


In [2]:
# all pantry ingredients used exactly
pantry = make_pantry({"chicken": (500.0, 5), "rice": (300.0, 7)})
meal_plan = [[make_recipe("Dish A", {"chicken": 500.0, "rice": 300.0})]]
score = get_ingredient_utilisation_score(meal_plan, pantry)

print(f"Full utilisation (expect 1.0): {score:.4f}")
assert score == 1.0

Full utilisation (expect 1.0): 1.0000


In [3]:
# only half of the pantry is consumed
pantry = make_pantry({"chicken": (400.0, 5), "rice": (400.0, 7)})
meal_plan = [
    [make_recipe("Dish A", {"chicken": 400.0})]  # rice untouched
]
score = get_ingredient_utilisation_score(meal_plan, pantry)

print(f"Half utilisation (expect 0.5): {score:.4f}")
assert score == 0.5

Half utilisation (expect 0.5): 0.5000


In [4]:
# recipes need ingredients not in the pantry - nothing consumed
pantry = make_pantry({"chicken": (500.0, 5)})
meal_plan = [[make_recipe("Dish A", {"broccoli": 200.0})]]
score = get_ingredient_utilisation_score(meal_plan, pantry)

print(f"No pantry overlap (expect 0.0): {score:.4f}")
assert score == 0.0

No pantry overlap (expect 0.0): 0.0000


In [5]:
# recipe needs more than available - capped at what's in the pantry
pantry = make_pantry({"chicken": (100.0, 5)})
meal_plan = [[make_recipe("Dish A", {"chicken": 999.0})]]
score = get_ingredient_utilisation_score(meal_plan, pantry)

print(f"Over-demand capped at stock (expect 1.0): {score:.4f}")
assert score == 1.0

Over-demand capped at stock (expect 1.0): 1.0000


In [6]:
# empty pantry returns 0
pantry = make_pantry({})
meal_plan = [[make_recipe("Dish A", {"chicken": 200.0})]]
score = get_ingredient_utilisation_score(meal_plan, pantry)

print(f"Empty pantry (expect 0.0): {score:.4f}")
assert score == 0.0

Empty pantry (expect 0.0): 0.0000


## 2. `get_expiry_weighted_utilisation_score`

Rewards consuming ingredients that are close to expiry more heavily. The raw score
(Σ `quantity_used × (1 / days_to_expiry)`) is normalised by total pantry stock, giving a result in [0, 1] - higher is better.

In [7]:
now = datetime.now()

# consuming an ingredient expiring in 1 day should score higher than one expiring in 7 days
pantry_urgent = make_pantry({"chicken": (100.0, 1)})
pantry_fresh = make_pantry({"chicken": (100.0, 7)})
meal_plan = [[make_recipe("Dish A", {"chicken": 100.0})]]

score_urgent = get_expiry_weighted_utilisation_score(meal_plan, pantry_urgent, now)
score_fresh = get_expiry_weighted_utilisation_score(meal_plan, pantry_fresh, now)

print(f"Urgent (1 day)  score: {score_urgent:.4f}")
print(f"Fresh  (7 days) score: {score_fresh:.4f}")
assert score_urgent > score_fresh, "Consuming near-expiry ingredients should yield a higher score"

Urgent (1 day)  score: 1.0000
Fresh  (7 days) score: 0.1429


In [8]:
# nothing consumed - score should be 0
pantry = make_pantry({"chicken": (500.0, 3)})
meal_plan = [[make_recipe("Dish A", {"broccoli": 200.0})]]
score = get_expiry_weighted_utilisation_score(meal_plan, pantry, now)

print(f"Nothing consumed (expect 0.0): {score:.4f}")
assert score == 0.0

Nothing consumed (expect 0.0): 0.0000


In [9]:
# manually verify the normalised formula:
# 200g consumed, 2 days until expiry → raw = 200 * (1/2) = 100.0; total_available = 200 → score = 100/200 = 0.5
pantry = make_pantry({"rice": (200.0, 2)})
meal_plan = [[make_recipe("Dish A", {"rice": 200.0})]]
score = get_expiry_weighted_utilisation_score(meal_plan, pantry, now)
expected = (200.0 * (1.0 / 2)) / 200.0

print(f"Manual check: score = {score:.4f}, expected = {expected:.4f}")
assert abs(score - expected) < 1e-6

Manual check: score = 0.5000, expected = 0.5000


In [10]:
# maximum score: all stock consumed and expiring in 1 day → raw/total = total/total = 1.0
pantry = make_pantry({"chicken": (300.0, 1)})
meal_plan = [[make_recipe("Dish A", {"chicken": 300.0})]]
score = get_expiry_weighted_utilisation_score(meal_plan, pantry, now)

print(f"All stock consumed, expiring in 1 day (expect 1.0): {score:.4f}")
assert score == 1.0

All stock consumed, expiring in 1 day (expect 1.0): 1.0000


## 3. `get_expiry_waste_score`

Measures the fraction of pantry stock that expires **within 7 days** but is **not consumed** by the meal plan.
Score is in [0, 1] - lower is better (less waste).

In [11]:
now = datetime.now()

# all near-expiry ingredients consumed -> nothing wasted -> score = 0.0
pantry = make_pantry({"chicken": (400.0, 2)})
meal_plan = [[make_recipe("Dish A", {"chicken": 400.0})]]
score = get_expiry_waste_score(meal_plan, pantry, now)

print(f"All near-expiry consumed (expect 0.0): {score:.4f}")
assert score == 0.0

All near-expiry consumed (expect 0.0): 0.0000


In [12]:
# near-expiry ingredient not consumed -> wasted fraction = 300 / 300 = 1.0
pantry = make_pantry({"broccoli": (300.0, 3)})
meal_plan = [[make_recipe("Dish A", {"rice": 200.0})]]  # broccoli not used
score = get_expiry_waste_score(meal_plan, pantry, now)

print(f"Near-expiry ingredient fully wasted (expect 1.0): {score:.4f}")
assert score == 1.0

Near-expiry ingredient fully wasted (expect 1.0): 1.0000


In [13]:
# ingredient expiring after 7 days and not consumed -> not counted as waste -> score = 0.0
pantry = make_pantry({"chicken": (500.0, 10)})
meal_plan = [[make_recipe("Dish A", {"rice": 200.0})]]  # chicken not used, but not near expiry
score = get_expiry_waste_score(meal_plan, pantry, now)

print(f"Ingredient expiring in 10 days, unused (expect 0.0): {score:.4f}")
assert score == 0.0

Ingredient expiring in 10 days, unused (expect 0.0): 0.0000


In [14]:
# partial consumption: 200g of 400g near-expiry stock consumed -> wasted = 200/400 = 0.5
pantry = make_pantry({"rice": (400.0, 4)})
meal_plan = [[make_recipe("Dish A", {"rice": 200.0})]]
score = get_expiry_waste_score(meal_plan, pantry, now)

print(f"Half of near-expiry stock consumed (expect 0.5): {score:.4f}")
assert abs(score - 0.5) < 1e-6

Half of near-expiry stock consumed (expect 0.5): 0.5000


## 4. `get_dietary_compliance`

Returns a score between 0 and 1.
- Hard constraints (vegan/vegetarian/gluten-free/lactose-free) immediately return **0.0** on any violation.
- Soft constraints (calorie and protein targets) return **1.0 - mean_relative_error**, so a perfect match gives 1.0.

In [15]:
# perfect match on calorie and protein targets -> score = 1.0
prefs = make_preferences(calorie_target=2000.0, protein_target=50.0)

# one day, one meal hitting the daily targets exactly
meal_plan = [[make_recipe("Dish A", {}, calories=2000.0, protein=50.0)]]
score = get_dietary_compliance(meal_plan, prefs)

print(f"Perfect nutritional match (expect 1.0): {score:.4f}")
assert score == 1.0

Perfect nutritional match (expect 1.0): 1.0000


In [16]:
# 50% deviation on both targets -> mean_relative_error = 0.5 -> score = 0.5
prefs = make_preferences(calorie_target=2000.0, protein_target=100.0)
meal_plan = [[make_recipe("Dish A", {}, calories=1000.0, protein=50.0)]]
score = get_dietary_compliance(meal_plan, prefs)

print(f"50%% deviation on both targets (expect 0.5): {score:.4f}")
assert abs(score - 0.5) < 1e-6

50%% deviation on both targets (expect 0.5): 0.5000


In [17]:
# hard constraint: vegetarian user with non-vegetarian recipe -> 0.0
prefs = make_preferences(is_vegetarian=True)
meal_plan = [[make_recipe("Steak", {}, calories=800.0, protein=60.0, tags=[])]]  # no VEGETARIAN tag
score = get_dietary_compliance(meal_plan, prefs)

print(f"Non-vegetarian recipe for vegetarian user (expect 0.0): {score:.4f}")
assert score == 0.0


Non-vegetarian recipe for vegetarian user (expect 0.0): 0.0000


In [18]:
# hard constraint: vegan user with vegetarian but not vegan recipe -> 0.0
prefs = make_preferences(is_vegetarian=True, is_vegan=True)
meal_plan = [[make_recipe("Veggie Dish", {}, calories=600.0, protein=20.0, tags=[DietaryTag.VEGETARIAN])]]
score = get_dietary_compliance(meal_plan, prefs)

print(f"Vegetarian recipe for vegan user (expect 0.0): {score:.4f}")
assert score == 0.0

Vegetarian recipe for vegan user (expect 0.0): 0.0000


In [19]:
# hard constraint: gluten-free user with non-gluten-free recipe -> 0.0
prefs = make_preferences(requires_gluten_free=True)
meal_plan = [[make_recipe("Pasta", {}, calories=700.0, protein=25.0, tags=[])]]
score = get_dietary_compliance(meal_plan, prefs)

print(f"Non-GF recipe for GF user (expect 0.0): {score:.4f}")
assert score == 0.0

Non-GF recipe for GF user (expect 0.0): 0.0000


In [20]:
# hard constraint satisfied: vegan user with vegan recipe, hitting targets -> 1.0
prefs = make_preferences(calorie_target=1800.0, protein_target=60.0, is_vegetarian=True, is_vegan=True)
meal_plan = [[make_recipe("Vegan Bowl", {}, calories=1800.0, protein=60.0, tags=[DietaryTag.VEGAN])]]
score = get_dietary_compliance(meal_plan, prefs)

print(f"Vegan recipe, perfect targets (expect 1.0): {score:.4f}")
assert score == 1.0

Vegan recipe, perfect targets (expect 1.0): 1.0000


In [21]:
# score is clamped to 0 when error exceeds 100%
prefs = make_preferences(calorie_target=2000.0, protein_target=50.0)
meal_plan = [[make_recipe("Empty Dish", {}, calories=0.0, protein=0.0)]]
score = get_dietary_compliance(meal_plan, prefs)

print(f"Zero nutrition against non-zero targets (expect 0.0): {score:.4f}")
assert score == 0.0

Zero nutrition against non-zero targets (expect 0.0): 0.0000


## 5. `get_budget_efficiency`

Returns a score between 0 and 1, measuring how efficiently the meal plan uses the weekly budget.
- If all ingredients come from the pantry (nothing to buy), returns **1.0**.
- If purchase cost ≤ budget, returns **1.0**.
- If purchase cost > budget, returns `budget / purchase_cost` (< 1.0).

In [22]:
# all ingredients already in pantry -> nothing to buy -> score = 1.0
pantry = make_pantry({"chicken": (500.0, 5)})
meal_plan = [[make_recipe("Dish A", {"chicken": 500.0})]]
score = get_budget_efficiency(meal_plan, pantry, {"chicken": 2.0}, weekly_budget=50.0)

print(f"All from pantry, nothing to buy (expect 1.0): {score:.4f}")
assert score == 1.0

All from pantry, nothing to buy (expect 1.0): 1.0000


In [23]:
# all ingredients must be bought, purchase cost == budget -> score = 1.0
# chicken costs €2.00/100g -> 100g costs €2.00
pantry = make_pantry({})
meal_plan = [[make_recipe("Dish A", {"chicken": 100.0})]]
score = get_budget_efficiency(meal_plan, pantry, {"chicken": 2.0}, weekly_budget=2.0)

print(f"Cost equals budget (expect 1.0): {score:.4f}")
assert score == 1.0

Cost equals budget (expect 1.0): 1.0000


In [24]:
# purchase cost = 2× budget -> efficiency = budget / cost = 0.5
# chicken: 100g at €2.00/100g = €2.00; budget = €1.00
pantry = make_pantry({})
meal_plan = [[make_recipe("Dish A", {"chicken": 100.0})]]
score = get_budget_efficiency(meal_plan, pantry, {"chicken": 2.0}, weekly_budget=1.0)

print(f"Cost is double budget (expect 0.5): {score:.4f}")
assert abs(score - 0.5) < 1e-6

Cost is double budget (expect 0.5): 0.5000


In [25]:
# partial pantry coverage: 200g available, 400g needed -> 200g to buy
# cost = (200 / 100) * €1.50 = €3.00; budget = €10.00 -> capped at 1.0
pantry = make_pantry({"rice": (200.0, 7)})
meal_plan = [[make_recipe("Dish A", {"rice": 400.0})]]
score = get_budget_efficiency(meal_plan, pantry, {"rice": 1.5}, weekly_budget=10.0)

print(f"Partial pantry cover, well within budget (expect 1.0): {score:.4f}")
assert score == 1.0

Partial pantry cover, well within budget (expect 1.0): 1.0000


In [26]:
# two ingredients, both fully purchased; total cost exceeds budget
# chicken: 200g × €2.00/100g = €4.00; rice: 300g × €1.00/100g = €3.00 -> total €7.00; budget €5.00
# efficiency = 5 / 7
pantry = make_pantry({})
meal_plan = [[make_recipe("Dish A", {"chicken": 200.0, "rice": 300.0})]]
score = get_budget_efficiency(meal_plan, pantry, {"chicken": 2.0, "rice": 1.0}, weekly_budget=5.0)
expected = 5.0 / 7.0

print(f"Two ingredients over budget (expect {expected:.4f}): {score:.4f}")
assert abs(score - expected) < 1e-6

Two ingredients over budget (expect 0.7143): 0.7143


In [27]:
# unknown ingredient cost falls back to default (€1.00/100g)
# 100g to buy at €1.00/100g = €1.00; budget = €0.50 -> efficiency = 0.5
pantry = make_pantry({})
meal_plan = [[make_recipe("Dish A", {"mystery_ingredient": 100.0})]]
score = get_budget_efficiency(meal_plan, pantry, {}, weekly_budget=0.5)

print(f"Unknown ingredient uses default cost €1/100g (expect 0.5): {score:.4f}")
assert abs(score - 0.5) < 1e-6

Unknown ingredient uses default cost €1/100g (expect 0.5): 0.5000


## 6. `get_purchase_dependency_score`

Measures what proportion of total required ingredient quantity is already available in the pantry (by weight). Result is in [0, 1] - higher means less purchasing needed.

In [28]:
# all needed ingredients in pantry -> nothing to purchase -> score = 1.0
pantry = make_pantry({"chicken": (500.0, 5), "rice": (300.0, 7)})
meal_plan = [[make_recipe("Dish A", {"chicken": 200.0, "rice": 100.0})]]
score = get_purchase_dependency_score(meal_plan, pantry)

print(f"All ingredients in pantry (expect 1.0): {score:.4f}")
assert score == 1.0

All ingredients in pantry (expect 1.0): 1.0000


In [29]:
# no ingredients in pantry -> everything must be purchased -> score = 0.0
pantry = make_pantry({})
meal_plan = [[make_recipe("Dish A", {"chicken": 300.0})]]
score = get_purchase_dependency_score(meal_plan, pantry)

print(f"Empty pantry, must buy everything (expect 0.0): {score:.4f}")
assert score == 0.0

Empty pantry, must buy everything (expect 0.0): 0.0000


In [30]:
# half the required quantity is in the pantry -> score = 0.5
# recipe needs 400g chicken; pantry has 200g -> pantry covers 200/400 = 0.5
pantry = make_pantry({"chicken": (200.0, 5)})
meal_plan = [[make_recipe("Dish A", {"chicken": 400.0})]]
score = get_purchase_dependency_score(meal_plan, pantry)

print(f"Half of required quantity in pantry (expect 0.5): {score:.4f}")
assert abs(score - 0.5) < 1e-6

Half of required quantity in pantry (expect 0.5): 0.5000


## 7. `get_variety_score`

Measures recipe variety as the number of unique recipe names divided by the total number of meal slots (`num_days × meals_per_day`). Result is in [0, 1] - higher means more variety.

In [31]:
# all 21 meal slots use a different recipe -> maximum variety -> score = 1.0
meal_plan = [[make_recipe(f"Recipe {i}", {})] for i in range(21)]
score = get_variety_score(meal_plan, num_days=7, meals_per_day=3)

print(f"All unique recipes (expect 1.0): {score:.4f}")
assert score == 1.0

All unique recipes (expect 1.0): 1.0000


In [32]:
# all 21 slots use the same recipe -> minimum variety -> score = 1/21
meal_plan = [[make_recipe("Chicken Rice", {})] for _ in range(21)]
score = get_variety_score(meal_plan, num_days=7, meals_per_day=3)
expected = 1 / 21

print(f"All same recipe (expect {expected:.4f}): {score:.4f}")
assert abs(score - expected) < 1e-6

All same recipe (expect 0.0476): 0.0476


In [33]:
# 7 unique recipes across 21 slots -> score = 7/21 ≈ 0.333
meal_plan = [[make_recipe(f"Recipe {i % 7}", {})] for i in range(21)]
score = get_variety_score(meal_plan, num_days=7, meals_per_day=3)
expected = 7 / 21

print(f"7 unique recipes across 21 slots (expect {expected:.4f}): {score:.4f}")
assert abs(score - expected) < 1e-6

7 unique recipes across 21 slots (expect 0.3333): 0.3333
